# Parallel Execution

*Notebook 14*

Run multiple agents concurrently to cut latency when tasks don't depend on each other.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner

# Notebook-specific imports
import asyncio
import time

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

When tasks are independent, running agents sequentially wastes time.

```text
sequential:  [agent 5s][agent 5s][agent 5s]   → 15s
```

Parallel execution runs all tasks concurrently.

Three 5-second agents finish in 5 seconds, not 15.

```text
parallel:    [agent 5s]
             [agent 5s]
             [agent 5s]                       → 5s
```

---

## ⚡ Async Recap

Two patterns power this notebook:

`await`: pauses a coroutine so others can run while it waits.

`asyncio.gather()`: runs several coroutines concurrently and waits for all of them.

By default, if any coroutine in `gather()` raises, the exception propagates immediately (Part 6 handles this).

---

## ⏱️ Part 1: Sequential vs Parallel: The Difference

**Sequential:** each agent waits for the previous to finish.
```
Agent A ──────────────────►
                            Agent B ──────────────────►
                                                       Agent C ──►
Total time: A + B + C
```

**Parallel:** all agents start at the same time.
```
Agent A ──────────────────►
Agent B ──────────────────►
Agent C ──────────────────►
Total time: max(A, B, C)
```

Use parallel when tasks are **independent**: no agent needs another's output to start.

---

## 🐢 Part 2: Sequential Baseline

First, let's see how long it takes to run three independent analyses sequentially.

In [ ]:
# Three independent analysis agents
market_agent = Agent(
    name="MarketAnalyst",
    instructions="You analyze market trends and opportunities. Be specific and concise.",
    model=MODEL
)

tech_agent = Agent(
    name="TechAnalyst",
    instructions="You analyze technical feasibility and implementation challenges. Be specific and concise.",
    model=MODEL
)

risk_agent = Agent(
    name="RiskAnalyst",
    instructions="You analyze risks and potential failure modes. Be specific and concise.",
    model=MODEL
)

topic = "Building an AI-powered personal finance app"

# --------------------------------------------------------------
print("✅ Three analysis agents ready")

#### Run Sequentially

In [ ]:
print("🐢 SEQUENTIAL EXECUTION")
print("=" * 60)

start = time.time()

market_result = await Runner.run(market_agent, input=topic)

tech_result = await Runner.run(tech_agent, input=topic)

risk_result = await Runner.run(risk_agent, input=topic)

sequential_time = time.time() - start

print(f"⏱️  Total time: {sequential_time:.1f}s")
print(f"\n📊 Market: {market_result.final_output}")
print(f"\n⚙️  Tech:   {tech_result.final_output}")
print(f"\n⚠️  Risk:   {risk_result.final_output}")
print("=" * 60)

---

## ⚡ Part 3: Parallel with `asyncio.gather()`

`asyncio.gather()` runs multiple coroutines concurrently.

It returns their results in the order they were passed in.

In your own code, the conversion is usually simple.

Take several independent `await Runner.run(...)` calls.

Place those same coroutines inside one `await asyncio.gather(...)` call.

In [ ]:
print("⚡ PARALLEL EXECUTION")
print("=" * 60)

start = time.time()

market_result, tech_result, risk_result = await asyncio.gather(
    Runner.run(market_agent, input=topic),
    Runner.run(tech_agent, input=topic),
    Runner.run(risk_agent, input=topic)
)

parallel_time = time.time() - start
speedup_ratio = sequential_time / parallel_time

print(f"⏱️  Total time: {parallel_time:.1f}s")
print(f"📉 Sequential / parallel wall-time ratio: {speedup_ratio:.1f}x")
print(f"\n📊 Market: {market_result.final_output}")
print(f"\n⚙️  Tech:   {tech_result.final_output}")
print(f"\n⚠️  Risk:   {risk_result.final_output}")
print("=" * 60)

This is one noisy comparison, not a benchmark.

API latency and output length vary between runs.

The code demonstrates overlap, and the measured ratio will vary.

---

## 🔀 Part 4: Parallel + Merge

Run specialists in parallel.

A synthesis agent merges their outputs into one final response.

This is the **fan-out / fan-in** pattern:

- Fan out: launch independent work in parallel

- Fan in: merge the results

In [ ]:
synthesis_instructions = (
    "You synthesize multiple expert analyses into a clear executive summary.\n"
    "Present the key points from each analysis and end with an overall recommendation."
)

synthesis_agent = Agent(
    name="SynthesisAgent",
    instructions=synthesis_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Synthesis agent ready")

#### Run Parallel + Synthesis

In [ ]:
print("🔀 PARALLEL + MERGE PIPELINE")
print("=" * 60)
print(f"Topic: {topic}\n")

start = time.time()

# Phase 1: Run all specialists in parallel
print("Phase 1: Running specialists in parallel...")

market_result, tech_result, risk_result = await asyncio.gather(
    Runner.run(market_agent, input=topic),
    Runner.run(tech_agent, input=topic),
    Runner.run(risk_agent, input=topic)
)

print(f"  ✓ All specialists complete ({time.time() - start:.1f}s)\n")

# Phase 2: Synthesize results
print("Phase 2: Synthesizing results...")
synthesis_input = (
    f"Synthesize these expert analyses on: {topic}\n\n"
    f"MARKET ANALYSIS:\n{market_result.final_output}\n\n"
    f"TECHNICAL ANALYSIS:\n{tech_result.final_output}\n\n"
    f"RISK ANALYSIS:\n{risk_result.final_output}"
)

synthesis_result = await Runner.run(synthesis_agent, input=synthesis_input)

total_time = time.time() - start

print(f"  ✓ Synthesis complete ({total_time:.1f}s total)")
print("\n" + "=" * 60)
print("📋 EXECUTIVE SUMMARY")
print("=" * 60)
print(synthesis_result.final_output)

### 💡 Why This Works

Fan-out makes the specialist phase roughly as long as the slowest specialist, plus overhead.

The synthesis step still runs afterward.

The synthesis agent merges the independent perspectives into one response.

---

## 💰 Part 5: Cost & Speed Tradeoffs

Parallelism changes scheduling, not the number of calls.

It can lower wall time when calls overlap.

It does not inherently reduce token use or cost.

More concurrent calls can pressure rate limits.

For some tasks, several small specialists can be faster and cheaper than one reasoning-model call.

That comes from model choice and workload, not concurrency alone.

<div style="text-align: left; display: inline-block;">

| | Sequential | Parallel |
|---|---|---|
| Latency | A + B + C | roughly max(A, B, C) |
| Token use | Comparable | Comparable |
| API calls | Same | Same |
| Use when | Tasks depend on each other | Tasks are independent |

</div>

---

## ⚠️ Part 6: When One Task Fails

By default, `asyncio.gather()` raises as soon as any task fails.

The exception propagates and you get no result list, even for tasks that succeeded.

The other tasks are not cancelled: they keep running in the background and still consume tokens.

The fix is one argument: `return_exceptions=True`.

In [ ]:
# ❌ Default behavior: one failure brings down the whole gather
async def analyze_topic(topic):
    agent = Agent(
        name="Analyst",
        instructions="Write two sentences analyzing this topic.",
        model=MODEL
    )

    result = await Runner.run(agent, input=f"Analyze: {topic}")

    return result.final_output

async def bad_task(topic):
    if topic == "SKIP":
        raise ValueError(f"Skipping topic: {topic}")
    return await analyze_topic(topic)

# --------------------------------------------------------------
topics = ["Python", "SKIP", "JavaScript"]
try:
    results = await asyncio.gather(*[bad_task(t) for t in topics])
    print(results)
except Exception as e:
    print(f"❌ gather raised: {e}")
    print("   No result list was returned")
    print("   Other unfinished tasks were not automatically cancelled")

### The Fix: `return_exceptions=True`

Exceptions are returned as values in the results list instead of raised.

You keep all results and handle failures explicitly.

In [ ]:
# ✅ return_exceptions=True: each result is either output or an Exception
results = await asyncio.gather(
    bad_task("Python"),
    bad_task("SKIP"),
    bad_task("JavaScript"),
    return_exceptions=True
)

for topic, result in zip(topics, results):
    if isinstance(result, Exception):
        print(f"⚠️  '{topic}' failed: {result}")
    else:
        print(f"✅ '{topic}': {result[:80]}")

### Partial Result Handling

With partial results you have three options depending on how critical the failed task is.

In [ ]:
# Using `results` from the previous cell, `topics` from the first demo

# Pattern: collect what succeeded, decide how to proceed
good = [(t, r) for t, r in zip(topics, results) if not isinstance(r, Exception)]
failed = [t for t, r in zip(topics, results) if isinstance(r, Exception)]

print(f"✅ {len(good)} succeeded, ⚠️  {len(failed)} failed")

if len(good) == 0:
    print("❌ All tasks failed: abort")
elif len(good) < 2:
    print("⚠️  Too few results: consider retrying failed tasks")
else:
    print("✅ Enough results: proceed with synthesis")
    combined = "\n\n".join(f"{t}: {r}" for t, r in good)

    partial_synthesis_agent = Agent(
        name="Synthesizer",
        instructions="Combine the provided analyses into a brief summary. Note any gaps.",
        model=MODEL
    )

    result = await Runner.run(partial_synthesis_agent, input=f"Synthesize (note: {failed} topics were unavailable):\n{combined}")

    print(f"\nSynthesis:\n{result.final_output[:300]}")

# In your own pipeline, replace the artificial `bad_task` list with
# `asyncio.gather(...)` of your real specialist agents from Part 4, and set
# the threshold based on which results are critical, e.g., abort if a
# required specialist failed, proceed if at least one of each category
# succeeded.

### 💡 Why This Works

`return_exceptions=True` returns exceptions alongside successful results.

The `isinstance(result, Exception)` check separates good from failures.

This prevents silent degradation: you always know which tasks failed and why.

---

## ⏱️ Part 7: Timeouts and Fallbacks

`return_exceptions=True` handles errors, but a slow agent can still hold up the whole `gather()`.

`asyncio.wait_for()` requests cancellation when a task exceeds its timeout.

In [ ]:
async def run_with_timeout(agent, message, timeout_seconds=10):
    """Run an agent with a timeout. Returns output or None if timed out."""
    try:
        result = await asyncio.wait_for(

            Runner.run(agent, input=message),

            timeout=timeout_seconds
        )
        return result.final_output
    except TimeoutError:
        return None

# --------------------------------------------------------------
print("✅ run_with_timeout() ready")

#### Run with Timeout

In [ ]:
slow_agent = Agent(
    name="SlowAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

# Call 1: Normal timeout, completes before the deadline
print("Call 1: 15-second timeout")
print("-" * 60)
timeout_result = await run_with_timeout(slow_agent, "What is 2 + 2?", timeout_seconds=15)

if timeout_result is None:
    print("⚠️ Agent timed out: using fallback response")
    fallback = "Analysis unavailable due to timeout."
    print(f"Fallback: {fallback}")
else:
    print(f"✅ Result: {timeout_result}")

# Call 2: Forced timeout, deadline too short for any API call to finish
print("\nCall 2: 0.001-second timeout (forced failure)")
print("-" * 60)
timeout_result = await run_with_timeout(slow_agent, "What is 2 + 2?", timeout_seconds=0.001)

if timeout_result is None:
    print("⚠️ Agent timed out: using fallback response")
    fallback = "Analysis unavailable due to timeout."
    print(f"Fallback: {fallback}")
else:
    print(f"✅ Result: {timeout_result}")

### 💡 Why This Works

When the timeout expires, `asyncio.wait_for()` cancels the local task.

It waits for cancellation before raising `TimeoutError`.

Catch that error to return a fallback.

Remote work already started may not be undone.

---

## 💪 Practice Exercises

### Exercise 1: Parallel Translation

*Covers: `asyncio.gather`, running agents in parallel*

Translate a piece of text into three languages concurrently.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Parallel Translation
# --------------------------------------------------------------
# Objective: Translate the same text to three languages in parallel.

text = "Artificial intelligence is changing how we build software."

# TODO 1: Create three translation agents: Spanish, French, Japanese
#           Each agent's instructions: translate text to [language] only

# TODO 2: Run all three in parallel using asyncio.gather()

# TODO 3: Print each translation with its language label
#           and the total elapsed time

# --- Write your code below this line ---

### Exercise 2: Parallel Analysis + Synthesis

*Covers: `asyncio.gather`, parallel analysis with synthesis*

Analyze a topic from three different angles concurrently, then merge the results.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Parallel Analysis + Synthesis
# --------------------------------------------------------------
# Objective: Three analysts run in parallel, one synthesizer merges.

analysis_topic = "The rise of remote work since 2020"

# TODO 1: Create three analyst agents with different lenses:
#           - Economic impact
#           - Social and cultural effects
#           - Technology and tools

# TODO 2: Run all three with asyncio.gather(..., return_exceptions=True)

# TODO 3: Separate successful outputs from exceptions
#         Continue if at least two perspectives succeeded
#         Otherwise, abort

# TODO 4: Create a synthesis agent
#         Pass successful outputs and missing lens names in its input
#         Run it, then print the final report and total elapsed time

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`asyncio.gather()` is the mechanism:**

- Pass multiple `Runner.run()` coroutines: they all start at the same time

- Results come back in the same order as the inputs
<br>
<br>

**Use parallel when tasks are independent:**

- No agent needs another's output to start

- Same number of API calls, with lower wall time when the calls overlap
<br>
<br>

**Fan-out / Fan-in is the full pattern:**

- Fan out: launch specialists in parallel

- Fan in: merge all outputs with a synthesis agent
<br>
<br>

**Always handle failures:**

- Default `gather()` raises on the first exception: no result list is returned, and the other tasks are not automatically cancelled

- `gather(..., return_exceptions=True)` returns exceptions as values: check each with `isinstance(result, Exception)`

- Decide: enough good results to continue, or abort?
<br>
<br>

**Timeouts limit slow tasks:**

- Use `asyncio.wait_for()` to add timeouts: decide whether to skip, use a default, or abort

---

## 📍 Next Step

**Notebook 15: Debate & Critique Pattern**  

Put agents in dialogue to challenge and revise an output.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-14-parallel-execution)

---